In [1]:
!pip install -q langchain transformers sentence-transformers datasets faiss-cpu langchain-community

In [2]:
!pip install -q langchain-text-splitters

In [3]:
!pip install -U langchain langchain-community langchain-core

In [4]:
!pip install -q langchainhub

In [5]:
# Step 1: Data Loading
from langchain_community.document_loaders import HuggingFaceDatasetLoader

dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "context"

loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)
data = loader.load()

# Dolly contains some empty contexts; let's filter those to save memory
data = [doc for doc in data if len(doc.page_content) > 0]
print(f"Loaded {len(data)} documents.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loaded 15011 documents.


In [6]:
#Step 2: Splitting for the "Context Window"
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
docs = text_splitter.split_documents(data)


In [7]:
#Step 3: Generating Embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

modelPath = "sentence-transformers/all-MiniLM-l6-v2"
embeddings = HuggingFaceEmbeddings(
    model_name=modelPath,
    model_kwargs={'device':'cpu'},
    encode_kwargs={'normalize_embeddings': False}
)

/tmp/ipykernel_15002/1697017914.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-l6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
#Step 4: The Vector Database (FAISS)
from langchain_community.vectorstores import FAISS

# This will take a minute as it encodes the text into math
db = FAISS.from_documents(docs, embeddings)

In [9]:
#Step 5: Preparing the Reader (LLM)
from transformers import AutoTokenizer, pipeline
from langchain_community.llms import HuggingFacePipeline

model_name = "Intel/dynamic_tinybert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
# We use a question-answering pipeline specifically
qa_pipeline = pipeline(
    "question-answering",
    model=model_name,
    tokenizer=tokenizer,
)

llm = HuggingFacePipeline(pipeline=qa_pipeline)

Invalid model-index. Not loading eval results into CardData.


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertForQuestionAnswering LOAD REPORT from: Intel/dynamic_tinybert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 
fit_dense.bias               | UNEXPECTED |  | 
fit_dense.weight             | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_15002/1221976012.py:14: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=qa_pipeline)


In [10]:
!pip install -q langchain-classic

In [14]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_community.llms import HuggingFacePipeline

# 1. On choisit un modèle de génération de texte (plus compatible avec LangChain)
# GPT2 est léger et parfait pour tester sans faire planter Google Colab
model_id = "gpt2"

# 2. Chargement du tokenizer et du modèle
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

# 3. Création du pipeline Hugging Face (Text-Generation)
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=100, # Longueur de la réponse
    device_map="auto"   # Utilise le GPU si disponible
)

# 4. On emballe le tout pour LangChain
llm = HuggingFacePipeline(pipeline=pipe)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
